# Figure S7 — effect of the total flow rate on the standard synthesis (S52)

Symmetric injection (5 equal flows), Q_tot = 6, 8, 10, 12, 14, 16 µL/s, sample 2 of each condition, AvaSpec-ULS4096CL-EVO.
Processing as in the original analysis: measurement CSV, dead-pixel patch, final spectrum = mean of the last max(5, 20 %) spectra,
Savitzky–Golay (21, 3) on 315–750 nm. Code: `figS07.py` (it also keeps the normalized overlay and the
6/10/14 µL/s kinetics plots of the original analysis: `plot_final(..., normalized=True)`, `plot_kinetics_spectra`, `plot_kinetics_extmax`).

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


### `io`

In [ ]:
"""Reading the instrument HDF5 files (both schemas)."""
from dataclasses import dataclass

import h5py
import numpy as np


@dataclass
class Sample:
    sample_id: str
    wavelengths: np.ndarray      # (n_wl,)
    absorbances: np.ndarray      # (n_spectra, n_wl), time-ordered
    flows_set: np.ndarray        # (5,) commanded flows, µL/s


def load_measurement_csv(path):
    """Instrument measurement CSV (`;`, header = timestamp + wavelengths, one spectrum per row).

    Returns (wavelengths, elapsed seconds since the first spectrum, absorbances (n_spectra, n_wl)).
    """
    import pandas as pd

    df = pd.read_csv(path, header=None, sep=";")
    wl = df.iloc[0, 1:].astype(float).to_numpy()
    times = pd.to_datetime(df.iloc[1:, 0])
    t_sec = (times - times.iloc[0]).dt.total_seconds().to_numpy()
    spectra = df.iloc[1:, 1:].apply(pd.to_numeric, errors="coerce").to_numpy()
    return wl, t_sec, spectra


def load_sample(path):
    """Load one sample H5.

    Format 2fast-sample 2.0. Returns None for a file without commanded flows.
    """
    with h5py.File(path, "r") as f:
        if "conditions/flow_set" not in f:
            return None
        flows = f["conditions/flow_set"][()]
        wl = f["spectra/wavelength"][()]
        absorb = f["spectra/extinction"][()]
        sample_id = str(f["sample"].attrs.get("sample_id", path.stem))
    absorb = np.asarray(absorb, dtype=float)
    if absorb.ndim == 1:
        absorb = absorb.reshape(1, -1)
    return Sample(sample_id, np.asarray(wl, dtype=float), absorb, np.asarray(flows, dtype=float))


### `preprocess`

In [ ]:
"""Spectrum preprocessing, identical to the analysis code used for the paper
(paperAnalysis/analyse_reproducibility/*.py). Parameters are scientific choices: do not change them.
"""
import numpy as np
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

ULS4096_PIXELS = 3198    # AvaSpec-ULS4096CL-EVO grid, 315–750 nm
USB2048_PIXELS = 1219    # AvaSpec-2048-USB2-RM grid, 310–1000 nm

# dead-pixel blocks of the ULS4096 (start, stop), filled with the value of the pixel just before
ULS4096_DEAD_PIXELS = ((1200, 1210), (1900, 1910), (2460, 2472))


def patch_dead_pixels(spectra):
    """ULS4096 dead-pixel blocks set to the value of the pixel just before (works on 1-D or (n_spectra, n_wl))."""
    s = np.array(spectra, dtype=float, copy=True)
    if s.shape[-1] == ULS4096_PIXELS:
        for start, stop in ULS4096_DEAD_PIXELS:
            s[..., start:stop] = s[..., start - 1:start]
    return s


def smooth(spectrum):
    """Spectrometer-specific dead-pixel patch + Savitzky–Golay smoothing."""
    s = patch_dead_pixels(spectrum)
    n = len(s)
    if n == ULS4096_PIXELS:
        return savgol_filter(s, window_length=101, polyorder=3)
    if n == USB2048_PIXELS:
        return savgol_filter(s, window_length=51, polyorder=3)
    w = min(51, n // 10)
    if w % 2 == 0:
        w += 1
    return savgol_filter(s, window_length=w, polyorder=3)


def smooth_all(absorbances):
    return np.array([smooth(row) for row in absorbances])


def final_spectrum(smoothed, n_tail):
    """'Final' spectrum = mean of the last `n_tail` smoothed spectra of the measurement."""
    n_tail = min(n_tail, smoothed.shape[0])
    return smoothed[-n_tail:].mean(axis=0)


def harmonize(wavelengths, spectrum, wl_min, wl_max, n_points):
    """Linear interpolation onto a common grid; zero outside the measured range."""
    target = np.linspace(wl_min, wl_max, n_points)
    mask = (wavelengths >= wl_min) & (wavelengths <= wl_max)
    f = interp1d(wavelengths[mask], spectrum[mask], kind="linear", bounds_error=False, fill_value=0.0)
    return target, f(target)


### `figS07`

In [ ]:
"""Figure S7 — effect of the total flow rate Q_tot on the final spectrum of the standard synthesis (S52, ULS4096).

Port of paperAnalysis/total flow variation/total_flow_variation_analysis.py, same data and processing.
NOTE: this figure uses its own processing (measurement CSV, Savitzky–Golay 21/3 on the 315–750 nm crop, final
spectrum = mean of the last max(5, 20 %) spectra), not the SG 101/3 used for Fig 2. Kept as-is (scientific choice).
"""
import numpy as np
from scipy.signal import savgol_filter


SESSION = "S52"
SAMPLE = "sample_2"
# manip folder -> total flow (µL/s), checked against pump_flows.csv in the original script.
# S52-M04 and S52-M01 repeat the conditions of _4 and _1 and are not plotted.
FLOW_MAP = {"S52-M07": 6, "S52-M06": 8, "S52-M05": 10,
            "S52-M03": 12, "S52-M02": 14, "S52-M08": 16}
KINETICS = ("S52-M07", "S52-M05", "S52-M02")   # 6, 10, 14 µL/s
WL_MIN, WL_MAX = 315, 750
SG_WINDOW, SG_ORDER = 21, 3
STEADY_FRAC, STEADY_MINROWS = 0.20, 5


def _smooth(y):
    y = np.asarray(y, dtype=float)
    w = SG_WINDOW + (SG_WINDOW % 2 == 0)
    w = min(w, len(y) if len(y) % 2 == 1 else len(y) - 1)
    return y if w <= SG_ORDER else savgol_filter(y, w, SG_ORDER)


def _steady_state(spectra):
    k = min(max(STEADY_MINROWS, int(np.ceil(len(spectra) * STEADY_FRAC))), len(spectra))
    return np.nanmean(spectra[-k:], axis=0)


def load(manip, raw=None):
    raw = RAW if raw is None else raw
    wl, t, spectra = load_measurement_csv(raw / SESSION / manip / f"measurement_spectra_{SAMPLE}.csv")
    return wl, t, patch_dead_pixels(spectra)


def compute(raw=None):
    mask_of = lambda wl: (wl >= WL_MIN) & (wl <= WL_MAX)
    final, kinetics = {}, {}
    for manip, flow in sorted(FLOW_MAP.items(), key=lambda kv: kv[1]):
        wl, t, spectra = load(manip, raw)
        m = mask_of(wl)
        y = _smooth(_steady_state(spectra)[m])
        final[manip] = dict(flow=flow, wl=wl[m], spectrum=y,
                            lambda_max=float(wl[m][np.argmax(y)]), ext_max=float(y.max()))
        if manip in KINETICS:
            s = np.array([_smooth(row) for row in spectra[:, m]])
            kinetics[manip] = dict(flow=flow, wl=wl[m], t=t, spectra=s, ext_max=np.nanmax(s, axis=1))
    return dict(final=final, kinetics=kinetics)


RC = {"font.size": 17, "axes.titlesize": 18, "axes.labelsize": 18, "xtick.labelsize": 15, "ytick.labelsize": 15,
      "legend.fontsize": 15, "legend.title_fontsize": 16, "figure.titlesize": 20, "axes.linewidth": 1.3,
      "xtick.direction": "in", "ytick.direction": "in", "xtick.major.size": 5, "ytick.major.size": 5}
KIN_COLORS = {"S52-M07": "#1F77B4", "S52-M05": "#2CA02C", "S52-M02": "#D62728"}


def _style(ax):
    import matplotlib.ticker as ticker
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(5))
    ax.tick_params(which="both", top=True, right=True)


def plot_final(result, normalized=False, title=True):
    import matplotlib.pyplot as plt
    with plt.rc_context(RC):
        items = list(result["final"].values())
        cmap = plt.get_cmap("viridis")
        fig, ax = plt.subplots(figsize=(7, 5.8))
        for i, d in enumerate(items):
            y = d["spectrum"] / np.nanmax(d["spectrum"]) if normalized else d["spectrum"]
            ax.plot(d["wl"], y, lw=2.0, color=cmap(i / (len(items) - 1)), label=f"{d['flow']} µL/s")
        ax.set_xlim(WL_MIN, WL_MAX)
        if normalized:
            ax.set_ylim(0, 1.05)
            ax.set_ylabel("Normalized extinction (a.u.)")
        else:
            ax.set_ylabel("Extinction (a.u.)")
        ax.set_xlabel("Wavelength (nm)")
        if title:
            ax.set_title("S52 sample 2 — final spectrum vs total flow")
        _style(ax)
        ax.legend(title="Total flow", loc="best")
        fig.tight_layout()
    return fig


def plot_kinetics_spectra(result, intermediate_alpha=0.25, title=True):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(8, 6.2))
        for manip in KINETICS:
            d = result["kinetics"][manip]
            n = len(d["t"])
            for i in range(n):
                last = i == n - 1
                ax.plot(d["wl"], d["spectra"][i], color=KIN_COLORS[manip], lw=2.6 if last else 1.3,
                        alpha=1.0 if last else intermediate_alpha, zorder=3 if last else 2)
        ax.set_xlim(WL_MIN, WL_MAX)
        ax.set_ylim(bottom=0)
        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Extinction (a.u.)")
        if title:
            ax.set_title("S52 sample 2 — spectral kinetics: 6 vs 10 vs 14 µL/s")
        _style(ax)
        handles = [Line2D([0], [0], color=KIN_COLORS[m], lw=2.6, label=f"{result['kinetics'][m]['flow']} µL/s (final)")
                   for m in KINETICS]
        handles.append(Line2D([0], [0], color="gray", lw=1.3, alpha=intermediate_alpha + 0.15, label="Intermediate spectra"))
        ax.legend(handles=handles, title="Total flow", loc="best")
        fig.tight_layout()
    return fig


def plot_kinetics_extmax(result, title=True):
    import matplotlib.pyplot as plt
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(7, 5.8))
        for manip in KINETICS:
            d = result["kinetics"][manip]
            ax.plot(d["t"], d["ext_max"], "o-", lw=2.0, ms=6, color=KIN_COLORS[manip], label=f"{d['flow']} µL/s")
        ax.set_xlabel("Elapsed time (s)")
        ax.set_ylabel("Extinction max (a.u.)")
        if title:
            ax.set_title("S52 sample 2 — extinction max kinetics: 6 vs 10 vs 14 µL/s")
        _style(ax)
        ax.legend(title="Total flow", loc="best")
        fig.tight_layout()
    return fig


## Figure

In [ ]:
result = compute()
for d in result['final'].values():
    print(f"{d['flow']:>3} µL/s  λmax {d['lambda_max']:6.1f} nm  Ext_max {d['ext_max']:.3f}")


In [ ]:
# Figure S7 = raw (non-normalized) final spectra, no title (decision 2026-10-12)
FIGURES.mkdir(exist_ok=True)
fig = plot_final(result, normalized=False, title=False)
for ext in ('png', 'pdf'):
    fig.savefig(FIGURES / f'figS07_total_flow.{ext}', dpi=300, bbox_inches='tight', facecolor='white')
